# EA2 – Ideación, diseño y documentación de la necesidad
**Proyecto integrador – Analítica de Datos** · Etapa 1: obtención y descripción básica del dataset

Flujo de este notebook: **obtener el dataset → describirlo brevemente → generar su resumen en `.txt`**.

**Necesidad (escenario hipotético):** la gerencia comercial de una cadena minorista necesita saber dónde se concentran las ventas (categorías, tiendas, periodos y segmentos de clientes) para orientar sus decisiones.

In [ ]:
# Importar las dos clases desde sus archivos .py.
# Si el notebook corre en Colab y los archivos no están, se descargan del repositorio.
import urllib.request
from pathlib import Path

BASE_REPO = "https://raw.githubusercontent.com/kedwinparga21-wq/Proyecto-integrado/main/"
for archivo in ("hf_dataset_loader.py", "dataset_summarizer.py"):
    if not Path(archivo).exists():
        urllib.request.urlretrieve(BASE_REPO + archivo, archivo)

import pandas as pd
from hf_dataset_loader import HFDatasetLoader
from dataset_summarizer import DatasetSummarizer

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
print("Clases importadas: HFDatasetLoader y DatasetSummarizer")

## 1. Obtención de datos
Se obtiene el dataset de Hugging Face mediante código y se carga en un DataFrame.

In [ ]:
cargador = HFDatasetLoader("infinite-dataset-hub/RetailSalesPerformance", split="train")
df = cargador.load()
print("Origen:", cargador.describe_source())

## 2. Descripción básica en Jupyter

**Algunos registros**

In [ ]:
df.head()

**Dimensiones** (filas y columnas)

In [ ]:
print("Filas:", df.shape[0])
print("Columnas:", df.shape[1])

**Nombres y tipos de las columnas**

In [ ]:
df.dtypes

**Valores nulos por columna**

In [ ]:
df.isna().sum()

**Rango de fechas y variedad de las variables clave**

In [ ]:
fechas = pd.to_datetime(df["Date"], errors="coerce")
print("Periodo:", fechas.min().date(), "a", fechas.max().date())
for columna in ["OutletName", "Location", "ProductCategory", "CustomerAgeGroup", "CustomerIncomeLevel", "Label"]:
    print(f"{columna}: {df[columna].nunique()} valores distintos")

## 3. ¿Qué representa cada fila y qué variables respaldan la necesidad?
- **Cada fila** es un registro de ventas: una tienda (`OutletName`, en una `Location`) vendió `UnitsSold` unidades de una `ProductCategory` en una `Date`, con unos `Revenue`, y se registra el perfil del cliente (`CustomerAgeGroup` y `CustomerIncomeLevel`).
- **Variables que respaldan la necesidad:** `ProductCategory`, `OutletName`, `Location`, `Date`, `UnitsSold`, `Revenue`, `CustomerAgeGroup` y `CustomerIncomeLevel` permiten comparar ventas por producto, lugar, tiempo y segmento de cliente.
- `Label` es una etiqueta de desempeño (High, Medium, Low) según la ficha del dataset.
- **Limitación:** el dataset no tiene costos, así que no permite hablar de rentabilidad, y es sintético (generado con IA).

## 4. Generación automática del resumen `.txt`
La clase `DatasetSummarizer` genera el archivo a partir del mismo DataFrame revisado en este notebook.

In [ ]:
resumidor = DatasetSummarizer(
    df,
    titulo="Resumen del DataFrame - EA2",
    origen=cargador.describe_source(),
    notas=["DataFrame completo, sin muestreo (split train).", "Dataset sintético según su ficha de Hugging Face."],
)
ruta_txt = resumidor.guardar("AE2/resultados/resumen_dataframe.txt")
print("Archivo generado en:", ruta_txt)
print()
print(ruta_txt.read_text(encoding="utf-8"))

In [ ]:
# Solo en Google Colab: descarga el resumen para subirlo al repositorio
try:
    from google.colab import files
    files.download(str(ruta_txt))
except ImportError:
    print("Fuera de Colab: el archivo está en", ruta_txt)